# 01 — Dataset Exploration, Spectral EDA & Leakage-Safe Validation Audit
### Enveda CASMI 2026 — Molecule ID From Mass Spectra

**Goal of this notebook:** understand the competition data end-to-end — schema, spectra,
molecular structures, duplication, leakage risk, candidate-space size, train/test shift —
and land on a **leakage-safe cross-validation strategy** and a **clean set of reusable
artifacts** for Notebook 02 (a spectral-retrieval baseline).

This notebook deliberately does **not** train a predictive model. It is analytical, not
merely descriptive: every major section ends with an `Observation / Why it matters /
Implication for modeling` block, and every finding is derived from the data actually
present in `../data/`, not assumed in advance.

> **Target:** ranked list of up to 25 candidate SMILES per test molecule.
> **Metric:** Mean Reciprocal Rank @ 25 (MRR@25).


## 1. Environment and imports

In [ ]:

import sys, os, re, json, math, time, warnings, random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

from scipy import stats as sp_stats
from scipy.spatial.distance import cosine as scipy_cosine

from sklearn.model_selection import GroupKFold

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# We deliberately do NOT globally silence warnings -- only specific, expected ones
# are filtered later, close to where they occur, so unexpected issues stay visible.

try:
    import rdkit
    from rdkit import Chem, RDLogger
    from rdkit.Chem import (
        AllChem, Descriptors, rdMolDescriptors, DataStructs, Draw
    )
    from rdkit.Chem.Scaffolds import MurckoScaffold
    RDLogger.DisableLog("rdApp.*")  # suppress per-molecule parse warnings; we track failures explicitly instead
    RDKIT_AVAILABLE = True
    print("RDKit version:", rdkit.__version__)
except ImportError as e:
    RDKIT_AVAILABLE = False
    print("RDKit not available:", e)

print("Python version:", sys.version)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
import sklearn
print("scikit-learn version:", sklearn.__version__)
import scipy
print("SciPy version:", scipy.__version__)
import matplotlib as mpl
print("Matplotlib version:", mpl.__version__)


## 2. Load the dataset

The competition data is known and fixed for this project, so we load it directly instead of
walking an unknown input directory. `train.parquet` and `test.parquet` live under `../data/`
relative to this notebook.

**`train.parquet` columns:** `ingest_lib, normalized_smiles, inchikey, inchikey14,
molecular_formula, ionization_mode, instrument_type, adduct, adduct_orig, precursor_mz,
precursor_error_ppm, ms2_mzs, ms2_normalized_intensities, num_peaks, base_peak_intensity,
collision_energy_ev, collision_energy_orig, collision_energy_orig_units`. `ms2_mzs` and
`ms2_normalized_intensities` are `list<double>` columns -- pyarrow decodes each cell straight
into a `numpy.ndarray`, so no peak-string parsing is needed. `collision_energy_ev` is itself a
list (ramped/stepped CE) and ~13% null. There is no `spectrum_id` / `molecule_id` column in
train; we mint a `spectrum_id` from the row index below.

**`test.parquet`** carries the same acquisition/spectral columns but identifies rows via
`molecule_id` / `spectrum_id` (multiple spectra can map to one target molecule) instead of
`inchikey14`, and has no `normalized_smiles` / `inchikey` / `molecular_formula` -- those are the
prediction target, so we never load or peek at them.


In [ ]:
DATA_DIR = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data"

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# train.parquet has no spectrum id of its own -- mint one from the row index.
train_df["spectrum_id"] = "train_" + train_df.index.astype(str)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("sample_submission shape:", sample_submission.shape)
print("unique inchikey14 (train):", train_df["inchikey14"].nunique())
print("unique inchikey (train):", train_df["inchikey"].nunique())
print("unique molecular_formula (train):", train_df["molecular_formula"].nunique())
if "molecule_id" in test_df.columns:
    print("unique molecule_id (test):", test_df["molecule_id"].nunique(),
          " unique spectrum_id (test):", test_df["spectrum_id"].nunique())
display(train_df.head())
display(test_df.head())


## 3. Dataset schema map

Maps the known raw columns onto the semantic roles the rest of this notebook uses
(`mol_col`, `spec_col`, `smiles_col`, ...). This is hardcoded from the real schema above --
no auto-detection needed.

`peaks_col` / `peaks_intensity_col` point at the two parallel `list<double>` columns that
together define a spectrum (fragment m/z and matching relative intensity); there is no single
combined "peaks" column in this dataset.

`molecule_id_col` (`inchikey14`) and `smiles_col` only resolve against `train_df` -- `test_df`
identifies rows by `molecule_id` instead and has no structure/label columns, which is
intentional (we never peek at test labels).


In [ ]:
DATA_MAP = {
    "molecule_id_col": "inchikey14",
    "spectrum_id_col": "spectrum_id",
    "smiles_col": "normalized_smiles",
    "precursor_mz_col": "precursor_mz",
    "peaks_col": "ms2_mzs",
    "peaks_intensity_col": "ms2_normalized_intensities",
    "adduct_col": "adduct",
    "ion_mode_col": "ionization_mode",
    "collision_energy_col": "collision_energy_ev",
    "instrument_col": "instrument_type",
    "formula_col": "molecular_formula",
}

for k, v in DATA_MAP.items():
    print(f"  {k:24s}: {v}")

print("""
TARGET            = molecular structure / SMILES
GROUPING UNIT     = molecule / connectivity  (this is what must not leak across CV folds)
OBSERVATION UNIT  = spectrum                 (one molecule can have many spectra)
""")


In [ ]:
def build_schema_table(df, source_name):
    rows = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        missing_pct = 100.0 * s.isna().mean() if n else np.nan
        try:
            nunique = s.nunique(dropna=True)
        except TypeError:
            # column holds unhashable values (e.g. numpy arrays / lists of spectral peaks);
            # coerce to a hashable representation just for the uniqueness count
            nunique = s.map(
                lambda x: tuple(x) if isinstance(x, (list, np.ndarray)) else x
            ).nunique(dropna=True)
        try:
            example_value = s.dropna().iloc[0] if s.notna().any() else None
        except TypeError:
            example_value = s.iloc[0] if n else None
        rows.append({
            "field": col,
            "source_file": source_name,
            "dtype": str(s.dtype),
            "missing_%": round(missing_pct, 2) if n else np.nan,
            "n_unique": nunique,
            "example_value": example_value,
        })
    return pd.DataFrame(rows)

schema_tables = []
if len(train_df):
    schema_tables.append(build_schema_table(train_df, "train.parquet"))
if len(test_df):
    schema_tables.append(build_schema_table(test_df, "test.parquet"))

schema_report = pd.concat(schema_tables, ignore_index=True) if schema_tables else pd.DataFrame()
schema_report


## 4. Basic train/test overview

A side-by-side comparison of train and test at the metadata level, before touching
structures or spectra in depth. We never infer or peek at test SMILES/labels here.


In [ ]:

def safe_nunique(df, col):
    return df[col].nunique(dropna=True) if (col and col in df.columns) else np.nan

def safe_value_counts(df, col, top=10):
    if col and col in df.columns:
        return df[col].value_counts(dropna=False).head(top)
    return pd.Series(dtype=int)

mol_col = DATA_MAP["molecule_id_col"]
spec_col = DATA_MAP["spectrum_id_col"]
smiles_col = DATA_MAP["smiles_col"]
mz_col = DATA_MAP["precursor_mz_col"]
adduct_col = DATA_MAP["adduct_col"]
ion_col = DATA_MAP["ion_mode_col"]
ce_col = DATA_MAP["collision_energy_col"]
inst_col = DATA_MAP["instrument_col"]

overview_rows = []
def add_stat(name, train_val, test_val):
    overview_rows.append({"statistic": name, "train": train_val, "test": test_val})

add_stat("n_rows", len(train_df), len(test_df))
add_stat("n_unique_molecules", safe_nunique(train_df, mol_col), safe_nunique(test_df, mol_col))
add_stat("n_unique_spectra", safe_nunique(train_df, spec_col), safe_nunique(test_df, spec_col))
add_stat("n_unique_raw_smiles", safe_nunique(train_df, smiles_col), safe_nunique(test_df, smiles_col))
add_stat("n_unique_adducts", safe_nunique(train_df, adduct_col), safe_nunique(test_df, adduct_col))
add_stat("n_unique_ion_modes", safe_nunique(train_df, ion_col), safe_nunique(test_df, ion_col))
add_stat("n_unique_collision_energies", safe_nunique(train_df, ce_col), safe_nunique(test_df, ce_col))
add_stat("n_unique_instruments", safe_nunique(train_df, inst_col), safe_nunique(test_df, inst_col))

if mz_col and mz_col in train_df.columns:
    add_stat("precursor_mz_min", train_df[mz_col].min(), test_df[mz_col].min() if mz_col in test_df.columns else np.nan)
    add_stat("precursor_mz_max", train_df[mz_col].max(), test_df[mz_col].max() if mz_col in test_df.columns else np.nan)
    add_stat("precursor_mz_mean", train_df[mz_col].mean(), test_df[mz_col].mean() if mz_col in test_df.columns else np.nan)

if mol_col and mol_col in train_df.columns and len(train_df):
    add_stat("spectra_per_molecule_mean", train_df.groupby(mol_col).size().mean(), np.nan)
    add_stat("spectra_per_molecule_median", train_df.groupby(mol_col).size().median(), np.nan)

overview_table = pd.DataFrame(overview_rows)
overview_table


In [ ]:

# Missingness comparison
def missingness_table(df, name):
    if not len(df):
        return pd.DataFrame()
    m = df.isna().mean().mul(100).round(2).rename(f"missing_%_{name}")
    return m

miss_train = missingness_table(train_df, "train")
miss_test = missingness_table(test_df, "test")
missingness_compare = pd.concat([miss_train, miss_test], axis=1) if (len(train_df) or len(test_df)) else pd.DataFrame()
missingness_compare


In [ ]:

# Train vs test plots for key observable variables (never using test labels, only
# observable/predictor columns that also exist at test time).
plot_cols = [c for c in [mz_col, ce_col] if c and c in train_df.columns]

for col in plot_cols:
    fig, ax = plt.subplots()
    if train_df[col].dtype.kind in "if":
        bins = 40
        ax.hist(train_df[col].dropna(), bins=bins, alpha=0.5, density=True, label="train")
        if col in test_df.columns:
            ax.hist(test_df[col].dropna(), bins=bins, alpha=0.5, density=True, label="test")
        ax.set_xlabel(col)
        ax.set_ylabel("density")
    ax.set_title(f"Train vs test: {col}")
    ax.legend()
    plt.show()

for col in [adduct_col, ion_col]:
    if col and col in train_df.columns:
        fig, ax = plt.subplots()
        vc_train = train_df[col].value_counts(normalize=True)
        vc_test = test_df[col].value_counts(normalize=True) if col in test_df.columns else pd.Series(dtype=float)
        combined = pd.DataFrame({"train": vc_train, "test": vc_test}).fillna(0)
        combined.plot(kind="bar", ax=ax)
        ax.set_title(f"Train vs test category frequency: {col}")
        ax.set_ylabel("proportion")
        plt.show()


## 5. Molecule-level analysis

Canonicalize every training SMILES with RDKit, and derive a rich set of structural
descriptors. We keep `raw_smiles`, `canonical_smiles`, and `connectivity_key` as
**separate** columns — we never overwrite the original label.

Because CASMI scoring is connectivity-based (after tautomer canonicalization), we also
attempt a tautomer-canonicalized `connectivity_key` (InChIKey14) when RDKit's tautomer
enumerator is available, falling back gracefully to a plain InChIKey14 otherwise.


In [ ]:

def canonicalize_smiles(smiles):
    """Return RDKit-canonical SMILES, or None if parsing fails."""
    if not RDKIT_AVAILABLE or smiles is None or (isinstance(smiles, float) and np.isnan(smiles)):
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


_tautomer_enumerator = None
def _get_tautomer_enumerator():
    global _tautomer_enumerator
    if _tautomer_enumerator is None and RDKIT_AVAILABLE:
        try:
            from rdkit.Chem.MolStandardize import rdMolStandardize
            _tautomer_enumerator = rdMolStandardize.TautomerEnumerator()
        except Exception:
            _tautomer_enumerator = False  # sentinel: unavailable
    return _tautomer_enumerator


def get_connectivity_key(smiles, use_tautomer_canon=True):
    """Best-effort competition-like connectivity key: InChIKey14 (first 14 chars),
    optionally after tautomer canonicalization if the installed RDKit supports it.
    Falls back to plain InChIKey14 (no tautomer canon) if unavailable.
    """
    if not RDKIT_AVAILABLE or smiles is None:
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    if use_tautomer_canon:
        enumerator = _get_tautomer_enumerator()
        if enumerator and enumerator is not False:
            try:
                mol = enumerator.Canonicalize(mol)
            except Exception:
                pass
    try:
        inchi = Chem.MolToInchi(mol)
        inchikey = Chem.InchiToInchiKey(inchi)
        if inchikey:
            return inchikey.split("-")[0]  # InChIKey14 = connectivity block
    except Exception:
        return None
    return None


def compute_molecule_descriptors(smiles):
    """Return a dict of structural descriptors for one SMILES, or all-NaN dict if invalid."""
    out = dict(
        canonical_smiles=None, inchikey=None, inchikey14=None, molecular_formula=None,
        exact_mass=np.nan, molecular_weight=np.nan, num_atoms=np.nan, num_heavy_atoms=np.nan,
        num_rings=np.nan, num_aromatic_rings=np.nan, num_heteroatoms=np.nan,
        num_rotatable_bonds=np.nan, formal_charge=np.nan,
    )
    if not RDKIT_AVAILABLE or smiles is None:
        return out
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return out
    try:
        out["canonical_smiles"] = Chem.MolToSmiles(mol, canonical=True)
        inchi = Chem.MolToInchi(mol)
        inchikey = Chem.InchiToInchiKey(inchi)
        out["inchikey"] = inchikey
        out["inchikey14"] = inchikey.split("-")[0] if inchikey else None
        out["molecular_formula"] = rdMolDescriptors.CalcMolFormula(mol)
        out["exact_mass"] = Descriptors.ExactMolWt(mol)
        out["molecular_weight"] = Descriptors.MolWt(mol)
        out["num_atoms"] = mol.GetNumAtoms()
        out["num_heavy_atoms"] = mol.GetNumHeavyAtoms()
        out["num_rings"] = rdMolDescriptors.CalcNumRings(mol)
        out["num_aromatic_rings"] = rdMolDescriptors.CalcNumAromaticRings(mol)
        out["num_heteroatoms"] = rdMolDescriptors.CalcNumHeteroatoms(mol)
        out["num_rotatable_bonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
        out["formal_charge"] = Chem.GetFormalCharge(mol)
    except Exception:
        pass
    return out

print("Reusable functions ready: canonicalize_smiles, get_connectivity_key, compute_molecule_descriptors")


In [ ]:

struct_df = pd.DataFrame()
if smiles_col and smiles_col in train_df.columns:
    struct_df = train_df[[c for c in [mol_col, smiles_col] if c]].drop_duplicates().reset_index(drop=True)
    struct_df = struct_df.rename(columns={smiles_col: "raw_smiles"})

    desc_records = [compute_molecule_descriptors(s) for s in struct_df["raw_smiles"]]
    desc_df = pd.DataFrame(desc_records)
    struct_df = pd.concat([struct_df, desc_df], axis=1)

    struct_df["connectivity_key"] = [
        get_connectivity_key(s, use_tautomer_canon=True) for s in struct_df["raw_smiles"]
    ]

    n_total = len(struct_df)
    n_invalid = struct_df["canonical_smiles"].isna().sum()
    print(f"Total distinct (molecule, raw_smiles) rows: {n_total}")
    print(f"Invalid / unparseable SMILES: {n_invalid} ({100*n_invalid/n_total:.2f}%)")
    print(f"Unique raw SMILES: {struct_df['raw_smiles'].nunique()}")
    print(f"Unique canonical SMILES: {struct_df['canonical_smiles'].nunique()}")
    print(f"Unique InChIKey14 / connectivity_key: {struct_df['connectivity_key'].nunique()}")

    n_raw = struct_df["raw_smiles"].nunique()
    n_canon = struct_df["canonical_smiles"].nunique()
    n_conn = struct_df["connectivity_key"].nunique()
    if n_raw > n_canon:
        print(f"\n-> {n_raw - n_canon} raw SMILES collapse onto fewer canonical SMILES "
              "(simple string-level duplication / differing atom order).")
    if n_canon > n_conn:
        print(f"-> {n_canon - n_conn} canonical SMILES further collapse onto fewer connectivity "
              "keys after tautomer canonicalization (tautomer-level duplication).")

    display(struct_df.head())
else:
    print("smiles_col not set in DATA_MAP -- fill in Section 3 before running this cell.")


## 6. Detect duplicate or repeated structures

This is one of the most important sections for validation design: if the same molecule
(connectivity) appears many times with different spectra, a naive random split will leak
that molecule's structure between train and validation.


In [ ]:

if len(struct_df):
    print("### A. Exact duplicate labels")
    for key in ["raw_smiles", "canonical_smiles", "connectivity_key"]:
        vc = struct_df[key].value_counts()
        n_dupe_groups = (vc > 1).sum()
        n_dupe_rows = vc[vc > 1].sum()
        print(f"  {key:18s}: {n_dupe_groups} duplicated groups covering {n_dupe_rows} rows "
              f"(out of {len(struct_df)})")


In [ ]:

if len(struct_df) and mol_col and mol_col in train_df.columns:
    # spectra_per_structure: join train_df (spectrum-level) back to connectivity_key
    conn_map = struct_df.set_index(mol_col)["connectivity_key"] if mol_col in struct_df.columns else None
    if conn_map is not None:
        spectra_conn = train_df[mol_col].map(conn_map)
        spectra_per_structure = spectra_conn.value_counts()

        print("### B. Spectra per molecule (by connectivity_key)")
        desc = spectra_per_structure.describe(percentiles=[0.5, 0.9, 0.95, 0.99])
        print(desc)

        fig, ax = plt.subplots()
        ax.hist(spectra_per_structure.values, bins=min(50, spectra_per_structure.nunique()))
        ax.set_xlabel("spectra per structure")
        ax.set_ylabel("number of structures")
        ax.set_title("Distribution: spectra per molecular structure")
        plt.show()

        print("\nTop 20 structures by number of spectra:")
        display(spectra_per_structure.head(20))
    else:
        print("Could not build connectivity map -- check mol_col / struct_df alignment.")
else:
    print("Skipping -- requires struct_df and molecule_id_col.")


In [ ]:

print("### C. Why a naive random spectrum-level split may leak")

if len(struct_df) and mol_col and mol_col in train_df.columns:
    rng = np.random.default_rng(SEED)
    n = len(train_df)
    idx = np.arange(n)
    rng.shuffle(idx)
    val_frac = 0.2
    n_val = int(n * val_frac)
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    conn_all = train_df[mol_col].map(conn_map)
    train_structs = set(conn_all.iloc[tr_idx].dropna())
    val_structs = conn_all.iloc[val_idx].dropna()

    leak_mask = val_structs.isin(train_structs)
    leak_pct = 100 * leak_mask.mean() if len(val_structs) else np.nan

    print(f"Simulated random {int(val_frac*100)}% spectrum-level holdout:")
    print(f"  Validation spectra: {len(val_idx)}")
    print(f"  Validation spectra whose molecule/connectivity ALSO appears in the "
          f"remaining training spectra: {leak_mask.sum()} ({leak_pct:.1f}%)")
    if leak_pct > 5:
        print("\n  -> This is a MEANINGFUL leakage rate. A model could partly succeed on "
              "validation simply by memorizing/retrieving spectra of molecules it has "
              "already seen, inflating validation MRR@25 relative to true test performance "
              "on structurally novel molecules.")
    else:
        print("\n  -> Leakage rate is low in this simulation, but grouped validation is "
              "still the safer default given the 'novel structure' nature of CASMI.")
else:
    print("Skipping -- requires struct_df and molecule_id_col.")


**Observation:** the leakage simulation above quantifies, on the real data, how many
naively-held-out spectra share a molecule with the training partition.

**Why it matters:** MRR@25 measured on a leaky split does not estimate performance on
truly unseen chemistry, which is what the hidden test set actually contains.

**Implication for modeling:** all future CV must group by `connectivity_key`, not split
at the spectrum level (see Section 21–22).


## 7. Molecular scaffold analysis (Bemis–Murcko)

In [ ]:

def get_murcko_scaffold(smiles):
    if not RDKIT_AVAILABLE or smiles is None:
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold, canonical=True)
    except Exception:
        return None

if len(struct_df):
    struct_df["murcko_scaffold"] = struct_df["canonical_smiles"].apply(get_murcko_scaffold)
    struct_df = struct_df.rename(columns={"murcko_scaffold": "scaffold_smiles"}) if False else struct_df
    struct_df["scaffold_smiles"] = struct_df["murcko_scaffold"]

    n_struct = struct_df["connectivity_key"].nunique()
    n_scaffold = struct_df["scaffold_smiles"].nunique()
    print(f"Unique structures: {n_struct}")
    print(f"Unique scaffolds:  {n_scaffold}")

    per_scaffold = struct_df.groupby("scaffold_smiles")["connectivity_key"].nunique().sort_values(ascending=False)
    singleton_frac = (per_scaffold == 1).mean()
    print(f"Fraction of singleton scaffolds (exactly 1 structure): {singleton_frac:.2%}")

    fig, ax = plt.subplots()
    ax.hist(per_scaffold.values, bins=min(50, per_scaffold.nunique()))
    ax.set_yscale("log")
    ax.set_xlabel("structures per scaffold")
    ax.set_ylabel("number of scaffolds (log scale)")
    ax.set_title("Scaffold-frequency distribution")
    plt.show()

    print("\nLargest scaffold families:")
    display(per_scaffold.head(10))
else:
    print("Skipping -- requires struct_df from Section 5.")


In [ ]:

if len(struct_df) and "scaffold_smiles" in struct_df.columns:
    per_scaffold = struct_df.groupby("scaffold_smiles")["connectivity_key"].nunique().sort_values(ascending=False)
    if len(per_scaffold):
        common_scaffold = per_scaffold.index[0]
        medium_scaffolds = per_scaffold[(per_scaffold >= 2) & (per_scaffold < per_scaffold.iloc[0])]
        medium_scaffold = medium_scaffolds.index[len(medium_scaffolds)//2] if len(medium_scaffolds) else None
        singleton_scaffold = per_scaffold[per_scaffold == 1].index[0] if (per_scaffold == 1).any() else None

        for label, scaf in [("very common", common_scaffold), ("medium-frequency", medium_scaffold), ("singleton", singleton_scaffold)]:
            if scaf is None:
                continue
            examples = struct_df[struct_df["scaffold_smiles"] == scaf][["raw_smiles", "canonical_smiles"]].head(5)
            print(f"\n{label.upper()} scaffold: {scaf}")
            display(examples)


**Observation:** scaffold frequency distribution and singleton fraction, computed above.

**Why it matters:** high singleton fraction implies most scaffolds are rare -> good test
of generalization but also means scaffold-based CV folds will be small-sample per scaffold.

**Implication for modeling:** use scaffold split as a *secondary, harder* validation on
top of the primary connectivity-grouped split.


## 8. Precursor m/z analysis

In [ ]:

if mz_col and mz_col in train_df.columns:
    mz_series = train_df[mz_col].dropna()
    print(mz_series.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(mz_series, bins=60)
    axes[0].set_title("Precursor m/z histogram (train)")
    axes[0].set_xlabel("precursor m/z")

    sorted_mz = np.sort(mz_series.values)
    ecdf_y = np.arange(1, len(sorted_mz) + 1) / len(sorted_mz)
    axes[1].plot(sorted_mz, ecdf_y)
    axes[1].set_title("Precursor m/z ECDF (train)")
    axes[1].set_xlabel("precursor m/z")
    axes[1].set_ylabel("cumulative fraction")
    plt.tight_layout()
    plt.show()

    if mz_col in test_df.columns:
        fig, ax = plt.subplots()
        ax.hist(mz_series, bins=60, alpha=0.5, density=True, label="train")
        ax.hist(test_df[mz_col].dropna(), bins=60, alpha=0.5, density=True, label="test")
        ax.set_title("Precursor m/z: train vs test")
        ax.legend()
        plt.show()

    if ion_col and ion_col in train_df.columns:
        fig, ax = plt.subplots()
        train_df.boxplot(column=mz_col, by=ion_col, ax=ax)
        ax.set_title("Precursor m/z by ion mode")
        plt.suptitle("")
        plt.show()

    if adduct_col and adduct_col in train_df.columns:
        top_adducts = train_df[adduct_col].value_counts().head(10).index
        fig, ax = plt.subplots(figsize=(10, 5))
        train_df[train_df[adduct_col].isin(top_adducts)].boxplot(column=mz_col, by=adduct_col, ax=ax, rot=45)
        ax.set_title("Precursor m/z by adduct (top 10)")
        plt.suptitle("")
        plt.show()
else:
    print("precursor_mz_col not set -- fill in DATA_MAP (Section 3).")


In [ ]:

# Common adduct mass offsets (Da), for sanity-checking precursor_mz - exact_mass
ADDUCT_OFFSETS = {
    "[M+H]+": 1.00728,
    "[M-H]-": -1.00728,
    "[M+Na]+": 22.98922,
    "[M+K]+": 38.96316,
    "[M+NH4]+": 18.03437,
    "[M+Cl]-": 34.96885,
    "[M+HCOO]-": 44.99820,
    "[M+CH3COO]-": 59.01385,
    "[M]+": 0.0,
    "[M]-": 0.0,
}

if mz_col and smiles_col and adduct_col and mol_col:
    merged = train_df[[mol_col, mz_col, adduct_col]].merge(
        struct_df[[mol_col, "exact_mass"]], on=mol_col, how="left"
    )
    merged["mz_minus_exact_mass"] = merged[mz_col] - merged["exact_mass"]

    offset_summary = merged.groupby(adduct_col)["mz_minus_exact_mass"].agg(["count", "mean", "std", "median"])
    offset_summary["expected_offset"] = offset_summary.index.map(ADDUCT_OFFSETS)
    offset_summary["diff_from_expected"] = offset_summary["mean"] - offset_summary["expected_offset"]
    display(offset_summary.sort_values("count", ascending=False))

    consistent = offset_summary.dropna(subset=["expected_offset"])
    if len(consistent):
        well_explained = (consistent["diff_from_expected"].abs() < 0.02).mean()
        print(f"\nOf adducts with a known theoretical offset, "
              f"{well_explained:.0%} have observed mean offset within 0.02 Da of theory.")
else:
    print("Skipping -- requires precursor_mz_col, smiles_col, adduct_col, molecule_id_col in DATA_MAP.")


**Observation:** measured `precursor_mz - exact_mass` per adduct, compared with the
textbook adduct offset.

**Why it matters:** if offsets are chemically consistent, precursor mass + adduct can be
used to reconstruct a tight neutral-mass window for candidate generation.

**Implication for modeling:** use this relationship in Notebook 02's mass-based candidate
filter (Section 20).


## 9. Ionization mode and adduct EDA

In [ ]:

for col, label in [(ion_col, "ion mode"), (adduct_col, "adduct")]:
    if col and col in train_df.columns:
        print(f"--- {label} ({col}) value counts (train) ---")
        vc = train_df[col].value_counts(dropna=False)
        vc_pct = (vc / vc.sum() * 100).round(2)
        display(pd.DataFrame({"count": vc, "pct": vc_pct}))

if ion_col and adduct_col and ion_col in train_df.columns and adduct_col in train_df.columns:
    print("--- ion_mode x adduct cross-tab (train) ---")
    display(pd.crosstab(train_df[ion_col], train_df[adduct_col]))


In [ ]:

for col, label in [(ion_col, "ion mode"), (adduct_col, "adduct")]:
    if col and col in train_df.columns and col in test_df.columns:
        train_set = set(train_df[col].dropna().unique())
        test_set = set(test_df[col].dropna().unique())
        only_train = train_set - test_set
        only_test = test_set - train_set
        print(f"{label}: categories only in TRAIN: {only_train if only_train else 'none'}")
        print(f"{label}: categories only in TEST:  {only_test if only_test else 'none'}")

        rare = train_df[col].value_counts()
        rare = rare[rare <= max(5, int(0.001 * len(train_df)))]
        print(f"{label}: rare train categories (<=~0.1% of rows): {list(rare.index)}\n")


## 10. Collision energy analysis

In [ ]:

if ce_col and ce_col in train_df.columns:
    ce_series = train_df[ce_col]
    print("dtype:", ce_series.dtype)
    print("missing %:", round(100 * ce_series.isna().mean(), 2))
    print(ce_series.describe() if ce_series.dtype.kind in "if" else ce_series.value_counts().head(20))

    if ce_series.dtype.kind in "if":
        fig, ax = plt.subplots()
        ax.hist(ce_series.dropna(), bins=40)
        ax.set_title("Collision energy distribution")
        ax.set_xlabel(ce_col)
        plt.show()

    if mol_col and mol_col in train_df.columns:
        ce_per_mol = train_df.groupby(mol_col)[ce_col].nunique()
        print("\nDistinct collision-energy values per molecule:")
        print(ce_per_mol.describe())

        multi_ce_mols = ce_per_mol[ce_per_mol > 1].index
        if len(multi_ce_mols):
            example_mol = multi_ce_mols[0]
            print(f"\nExample molecule with multiple collision energies: {example_mol}")
            display(train_df[train_df[mol_col] == example_mol][[mol_col, ce_col] + ([mz_col] if mz_col else [])])
else:
    print("collision_energy_col not set -- fill in DATA_MAP (Section 3).")


## 11. Spectrum-level EDA

Reusable `plot_spectrum` function used throughout the rest of the notebook.


In [ ]:

def get_spectrum_peaks(row_or_id):
    """Return (mz, intensity) for a given train_df/test_df row: `peaks_col` and
    `peaks_intensity_col` are already parallel `list<double>` arrays (pyarrow decodes them
    straight into numpy arrays), so this is a direct lookup -- no string parsing needed.
    """
    if isinstance(row_or_id, pd.Series):
        row = row_or_id
        mz_field, int_field = DATA_MAP["peaks_col"], DATA_MAP["peaks_intensity_col"]
        if mz_field in row.index and row[mz_field] is not None:
            mz = np.asarray(row[mz_field], dtype=float)
            intensity = np.asarray(row[int_field], dtype=float) if (int_field in row.index and row[int_field] is not None) else np.array([])
            return mz, intensity
    return np.array([]), np.array([])


def plot_spectrum(mz, intensity, title=None, precursor_mz=None, collision_energy=None,
                   adduct=None, ion_mode=None, max_mz=None, normalize=True, ax=None):
    """Stem plot of a single spectrum: fragment m/z (x) vs relative intensity (y)."""
    mz = np.asarray(mz, dtype=float)
    intensity = np.asarray(intensity, dtype=float)
    if max_mz is not None:
        keep = mz <= max_mz
        mz, intensity = mz[keep], intensity[keep]
    if normalize and len(intensity) and intensity.max() > 0:
        intensity = intensity / intensity.max()

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(9, 4))

    if len(mz):
        ax.vlines(mz, 0, intensity, linewidth=1)
        ax.scatter(mz, intensity, s=8)
    else:
        ax.text(0.5, 0.5, "no peaks", ha="center", va="center", transform=ax.transAxes)

    if precursor_mz is not None:
        ax.axvline(precursor_mz, color="red", linestyle="--", linewidth=1, label=f"precursor {precursor_mz:.4f}")
        ax.legend(fontsize=8)

    subtitle_bits = []
    if adduct is not None:
        subtitle_bits.append(f"adduct={adduct}")
    if ion_mode is not None:
        subtitle_bits.append(f"mode={ion_mode}")
    if collision_energy is not None:
        subtitle_bits.append(f"CE={collision_energy}")
    subtitle = "  |  ".join(subtitle_bits)

    full_title = title or ""
    if subtitle:
        full_title = f"{full_title}\n{subtitle}" if full_title else subtitle
    ax.set_title(full_title, fontsize=10)
    ax.set_xlabel("fragment m/z")
    ax.set_ylabel("relative intensity" if normalize else "intensity")

    if own_fig:
        plt.tight_layout()
        plt.show()

print("plot_spectrum() ready.")


## 12. Peak statistics

In [ ]:

def compute_spectrum_features(mz, intensity, precursor_mz=None):
    mz = np.asarray(mz, dtype=float)
    intensity = np.asarray(intensity, dtype=float)
    n = len(mz)
    out = dict(n_peaks=n, mz_min=np.nan, mz_max=np.nan, mz_mean=np.nan, mz_std=np.nan,
               intensity_max=np.nan, intensity_mean=np.nan, intensity_median=np.nan,
               base_peak_mz=np.nan, fraction_peaks_below_100=np.nan,
               fraction_peaks_below_200=np.nan, fraction_intensity_top1=np.nan,
               fraction_intensity_top5=np.nan, fragment_mass_range=np.nan)
    if n == 0:
        return out
    out["mz_min"] = mz.min()
    out["mz_max"] = mz.max()
    out["mz_mean"] = mz.mean()
    out["mz_std"] = mz.std()
    out["intensity_max"] = intensity.max()
    out["intensity_mean"] = intensity.mean()
    out["intensity_median"] = np.median(intensity)
    out["base_peak_mz"] = mz[np.argmax(intensity)]
    out["fraction_peaks_below_100"] = np.mean(mz < 100)
    out["fraction_peaks_below_200"] = np.mean(mz < 200)
    total_int = intensity.sum()
    if total_int > 0:
        sorted_int = np.sort(intensity)[::-1]
        out["fraction_intensity_top1"] = sorted_int[:1].sum() / total_int
        out["fraction_intensity_top5"] = sorted_int[:5].sum() / total_int
    out["fragment_mass_range"] = mz.max() - mz.min()
    return out

print("compute_spectrum_features() ready.")

if DATA_MAP["peaks_col"] and DATA_MAP["peaks_col"] in train_df.columns:
    feats = []
    for _, row in train_df.iterrows():
        mz, inten = get_spectrum_peaks(row)
        feats.append(compute_spectrum_features(mz, inten, row.get(mz_col)))
    peak_feat_df = pd.DataFrame(feats)
    train_spectrum_features = pd.concat([train_df.reset_index(drop=True), peak_feat_df], axis=1)

    display(peak_feat_df.describe())

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0,0].hist(peak_feat_df["n_peaks"].dropna(), bins=40); axes[0,0].set_title("n_peaks")
    axes[0,1].hist(peak_feat_df["fragment_mass_range"].dropna(), bins=40); axes[0,1].set_title("fragment_mass_range")
    axes[1,0].hist(peak_feat_df["base_peak_mz"].dropna(), bins=40); axes[1,0].set_title("base_peak_mz")
    axes[1,1].hist(peak_feat_df["fraction_intensity_top1"].dropna(), bins=40); axes[1,1].set_title("fraction_intensity_top1")
    plt.tight_layout()
    plt.show()
else:
    train_spectrum_features = pd.DataFrame()
    print("peaks_col not set in DATA_MAP -- populate it (or adapt get_spectrum_peaks) before running this cell.")


In [ ]:

# Train vs test comparison of peak-derived features requires peaks to also be available
# for test (they normally are -- spectra are the *input*, SMILES are the *target*).
if len(train_spectrum_features) and DATA_MAP["peaks_col"] in test_df.columns:
    test_feats = []
    for _, row in test_df.iterrows():
        mz, inten = get_spectrum_peaks(row)
        test_feats.append(compute_spectrum_features(mz, inten, row.get(mz_col)))
    test_peak_feat_df = pd.DataFrame(test_feats)

    for col in ["n_peaks", "fragment_mass_range", "base_peak_mz"]:
        fig, ax = plt.subplots()
        ax.hist(peak_feat_df[col].dropna(), bins=40, alpha=0.5, density=True, label="train")
        ax.hist(test_peak_feat_df[col].dropna(), bins=40, alpha=0.5, density=True, label="test")
        ax.set_title(f"Train vs test: {col}")
        ax.legend()
        plt.show()


## 13. Spectrum examples

Eight representative spectra spanning low/high mass, few/many peaks, both ion modes,
and common/unusual adducts.


In [ ]:

if len(train_spectrum_features):
    df_ex = train_spectrum_features.copy()
    picks = {}

    if mz_col in df_ex.columns:
        picks["low precursor mass"] = df_ex.loc[df_ex[mz_col].idxmin()]
        picks["high precursor mass"] = df_ex.loc[df_ex[mz_col].idxmax()]
    if "n_peaks" in df_ex.columns:
        nonzero = df_ex[df_ex["n_peaks"] > 0]
        if len(nonzero):
            picks["few peaks"] = nonzero.loc[nonzero["n_peaks"].idxmin()]
            picks["many peaks"] = nonzero.loc[nonzero["n_peaks"].idxmax()]
    if ion_col and ion_col in df_ex.columns:
        modes = df_ex[ion_col].dropna().unique()
        for m in modes[:2]:
            picks[f"ion mode = {m}"] = df_ex[df_ex[ion_col] == m].iloc[0]
    if adduct_col and adduct_col in df_ex.columns:
        vc = df_ex[adduct_col].value_counts()
        if len(vc):
            picks["common adduct"] = df_ex[df_ex[adduct_col] == vc.index[0]].iloc[0]
        if len(vc) > 1:
            picks["unusual adduct"] = df_ex[df_ex[adduct_col] == vc.index[-1]].iloc[0]

    n = len(picks)
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, (label, row) in zip(axes, picks.items()):
        mz, inten = get_spectrum_peaks(row)
        title_bits = [f"{label}"]
        if mol_col:
            title_bits.append(f"mol={row.get(mol_col)}")
        plot_spectrum(
            mz, inten, title=" | ".join(title_bits),
            precursor_mz=row.get(mz_col), collision_energy=row.get(ce_col),
            adduct=row.get(adduct_col), ion_mode=row.get(ion_col), ax=ax,
        )
        info_cols = [c for c in [mol_col, smiles_col, mz_col, adduct_col, ion_col, ce_col, "n_peaks"] if c and c in row.index]
        print(f"--- {label} ---")
        print(row[info_cols])
        print()

    for ax in axes[len(picks):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping -- requires train_spectrum_features from Section 12.")


## 14. Same molecule, different spectrum

How variable is the observed spectrum for exactly the same molecular structure across
different acquisition settings?


In [ ]:

if len(train_spectrum_features) and mol_col:
    counts = train_spectrum_features[mol_col].value_counts()
    repeated_mols = counts[counts >= 2].index

    varying_cols = [c for c in [ce_col, adduct_col, ion_col] if c and c in train_spectrum_features.columns]
    chosen = []
    for m in repeated_mols:
        sub = train_spectrum_features[train_spectrum_features[mol_col] == m]
        if varying_cols and sub[varying_cols].drop_duplicates().shape[0] > 1:
            chosen.append(m)
        if len(chosen) >= 5:
            break
    if not chosen:
        chosen = list(repeated_mols[:5])

    for m in chosen:
        sub = train_spectrum_features[train_spectrum_features[mol_col] == m].head(4)
        smi = sub[smiles_col].iloc[0] if smiles_col in sub.columns else None
        print(f"Molecule {m}  |  canonical SMILES candidate: {smi}")

        fig, axes = plt.subplots(1, len(sub), figsize=(5 * len(sub), 4))
        axes = np.atleast_1d(axes)
        for ax, (_, row) in zip(axes, sub.iterrows()):
            mz, inten = get_spectrum_peaks(row)
            plot_spectrum(mz, inten, precursor_mz=row.get(mz_col), collision_energy=row.get(ce_col),
                          adduct=row.get(adduct_col), ion_mode=row.get(ion_col), ax=ax)
        plt.tight_layout()
        plt.show()
else:
    print("Skipping -- requires train_spectrum_features and molecule_id_col.")


**Observation:** visual variability of repeated spectra for the same molecule under
different collision energy / adduct / ion mode.

**Why it matters:** if within-molecule spectral variability is high, a naive nearest-neighbor
retrieval baseline needs to be robust to acquisition condition, not just structure.

**Implication for modeling:** consider normalizing or conditioning spectral similarity on
collision energy / adduct in Notebook 02.


## 15. Similar molecules, different spectra (Morgan / Tanimoto)

In [ ]:

def get_morgan_fp(smiles, radius=2, n_bits=2048):
    if not RDKIT_AVAILABLE or smiles is None:
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    try:
        from rdkit.Chem import rdFingerprintGenerator
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
        return gen.GetFingerprint(mol)
    except Exception:
        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)

if len(struct_df):
    valid_struct = struct_df.dropna(subset=["canonical_smiles"]).drop_duplicates("connectivity_key").reset_index(drop=True)
    valid_struct["morgan_fp"] = valid_struct["canonical_smiles"].apply(get_morgan_fp)
    print(f"Computed Morgan fingerprints for {valid_struct['morgan_fp'].notna().sum()} structures.")
else:
    valid_struct = pd.DataFrame()
    print("Skipping -- requires struct_df.")


In [ ]:

if len(valid_struct) >= 2:
    fps = valid_struct["morgan_fp"].tolist()
    smis = valid_struct["canonical_smiles"].tolist()
    masses = valid_struct["exact_mass"].tolist()

    # Sample-limited all-pairs search to keep this tractable for large datasets
    MAX_MOLS_FOR_PAIR_SEARCH = 800
    rng = np.random.default_rng(SEED)
    if len(fps) > MAX_MOLS_FOR_PAIR_SEARCH:
        sample_idx = rng.choice(len(fps), MAX_MOLS_FOR_PAIR_SEARCH, replace=False)
    else:
        sample_idx = np.arange(len(fps))

    best_pairs = []
    for ii, i in enumerate(sample_idx):
        if fps[i] is None:
            continue
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], [fps[j] for j in sample_idx[ii+1:] if fps[j] is not None])
        js = [j for j in sample_idx[ii+1:] if fps[j] is not None]
        for sim, j in zip(sims, js):
            if 0.7 <= sim < 1.0:  # similar but not identical
                best_pairs.append((sim, i, j))

    best_pairs.sort(reverse=True)
    print(f"Found {len(best_pairs)} high-similarity (0.7-1.0) non-identical pairs in the sample.")

    for sim, i, j in best_pairs[:5]:
        print(f"\nTanimoto = {sim:.3f}")
        print(f"  SMILES A: {smis[i]}   exact_mass={masses[i]:.4f}")
        print(f"  SMILES B: {smis[j]}   exact_mass={masses[j]:.4f}")
else:
    print("Skipping -- need at least 2 valid structures with fingerprints.")


## 16. Same or near-identical mass, different structure

This directly demonstrates why precursor mass alone cannot solve CASMI: many distinct
structures share (almost) the same exact mass.


In [ ]:

if len(struct_df):
    valid = struct_df.dropna(subset=["exact_mass", "connectivity_key"]).drop_duplicates("connectivity_key")
    valid = valid.sort_values("exact_mass").reset_index(drop=True)

    MASS_TOL = 0.01
    near_mass_groups = []
    i = 0
    masses = valid["exact_mass"].values
    n = len(valid)
    while i < n:
        j = i
        while j + 1 < n and masses[j+1] - masses[i] < MASS_TOL:
            j += 1
        if j > i:
            near_mass_groups.append(valid.iloc[i:j+1])
        i = j + 1

    print(f"Found {len(near_mass_groups)} groups of structures within {MASS_TOL} Da of each other "
          f"(and with distinct connectivity).")

    for grp in near_mass_groups[:5]:
        print("\n" + "=" * 80)
        display(grp[["canonical_smiles", "molecular_formula", "exact_mass"]])
        if RDKIT_AVAILABLE and len(grp) >= 2:
            fp_a = get_morgan_fp(grp.iloc[0]["canonical_smiles"])
            fp_b = get_morgan_fp(grp.iloc[1]["canonical_smiles"])
            if fp_a is not None and fp_b is not None:
                sim = DataStructs.TanimotoSimilarity(fp_a, fp_b)
                print(f"Morgan similarity between first two entries: {sim:.3f}")
else:
    print("Skipping -- requires struct_df.")


**Observation:** groups of structurally distinct molecules within 0.01 Da of one
another, with (often) low Morgan similarity despite near-identical mass.

**Why it matters:** confirms precursor mass is necessary but not sufficient — it narrows
candidates but does not resolve structure on its own.

**Implication for modeling:** mass filtering should be paired with spectral similarity /
fragmentation-pattern reranking, not used alone.


## 17. Neutral-loss exploration

In [ ]:

COMMON_LOSSES = {"H2O": 18.0106, "CO": 27.9949, "CO2": 43.9898, "NH3": 17.0265}

if len(train_spectrum_features) and mz_col in train_spectrum_features.columns:
    all_losses = []
    for _, row in train_spectrum_features.iterrows():
        prec = row.get(mz_col)
        if pd.isna(prec):
            continue
        mz, _ = get_spectrum_peaks(row)
        if len(mz):
            all_losses.extend((prec - mz).tolist())
    all_losses = np.array(all_losses)
    all_losses = all_losses[all_losses > 0]

    print(f"Computed {len(all_losses)} neutral-loss values from precursor - fragment m/z.")

    fig, ax = plt.subplots()
    ax.hist(all_losses, bins=200, range=(0, min(200, all_losses.max() if len(all_losses) else 200)))
    for name, val in COMMON_LOSSES.items():
        ax.axvline(val, color="red", linestyle="--", alpha=0.6)
        ax.text(val, ax.get_ylim()[1]*0.9, name, rotation=90, fontsize=8, color="red")
    ax.set_xlabel("neutral loss (Da)")
    ax.set_ylabel("count")
    ax.set_title("Neutral-loss distribution with common chemical losses marked")
    plt.show()

    TOL = 0.02
    for name, val in COMMON_LOSSES.items():
        frac = np.mean(np.abs(all_losses - val) < TOL) if len(all_losses) else np.nan
        print(f"  Fraction of neutral losses within {TOL} Da of {name} ({val}): {frac:.4%}")
else:
    print("Skipping -- requires train_spectrum_features with precursor_mz.")


## 18. Peak-frequency analysis

In [ ]:

if len(train_spectrum_features) and mz_col in train_spectrum_features.columns:
    BIN_WIDTH = 0.01
    all_mz = []
    all_ratio = []
    for _, row in train_spectrum_features.iterrows():
        prec = row.get(mz_col)
        mz, _ = get_spectrum_peaks(row)
        if len(mz):
            all_mz.extend(mz.tolist())
            if pd.notna(prec) and prec > 0:
                all_ratio.extend((mz / prec).tolist())
    all_mz = np.array(all_mz)
    all_ratio = np.array(all_ratio)

    if len(all_mz):
        bins = np.arange(0, all_mz.max() + BIN_WIDTH, BIN_WIDTH)
        binned = pd.cut(all_mz, bins=bins)
        freq = pd.Series(binned).value_counts().sort_values(ascending=False)
        print("Most frequent fragment m/z bins across all training spectra:")
        display(freq.head(20))

    if len(all_ratio):
        fig, ax = plt.subplots()
        ax.hist(all_ratio, bins=100, range=(0, 1.05))
        ax.set_xlabel("fragment m/z / precursor m/z")
        ax.set_ylabel("count")
        ax.set_title("Normalized fragment m/z distribution")
        plt.show()
else:
    print("Skipping -- requires train_spectrum_features with precursor_mz.")


## 19. Spectrum similarity baseline analysis

Do spectra from the *same* molecule look more similar to each other than spectra from
*different* molecules? This tells us how promising a retrieval baseline is for Notebook 02.


In [ ]:

def bin_spectrum(mz, intensity, mz_max=1000, bin_width=1.0):
    n_bins = int(mz_max / bin_width) + 1
    vec = np.zeros(n_bins)
    if len(mz) == 0:
        return vec
    idx = np.clip((mz / bin_width).astype(int), 0, n_bins - 1)
    np.add.at(vec, idx, intensity)
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

def compute_spectrum_similarity(mz1, int1, mz2, int2, mz_max=1000, bin_width=1.0):
    v1 = bin_spectrum(mz1, int1, mz_max, bin_width)
    v2 = bin_spectrum(mz2, int2, mz_max, bin_width)
    denom = (np.linalg.norm(v1) * np.linalg.norm(v2))
    if denom == 0:
        return 0.0
    return float(np.dot(v1, v2) / denom)

print("compute_spectrum_similarity() ready.")

if len(train_spectrum_features) and mol_col in train_spectrum_features.columns:
    binned_vecs = [
        bin_spectrum(*get_spectrum_peaks(row)) for _, row in train_spectrum_features.iterrows()
    ]
    mol_ids = train_spectrum_features[mol_col].values

    rng = np.random.default_rng(SEED)
    N_PAIRS = 3000

    same_sims, diff_sims = [], []
    n = len(binned_vecs)
    mol_to_indices = defaultdict(list)
    for i, m in enumerate(mol_ids):
        mol_to_indices[m].append(i)
    same_mol_candidates = [m for m, idxs in mol_to_indices.items() if len(idxs) >= 2]

    for _ in range(N_PAIRS):
        if same_mol_candidates and rng.random() < 0.5:
            m = same_mol_candidates[rng.integers(len(same_mol_candidates))]
            i, j = rng.choice(mol_to_indices[m], 2, replace=False)
            v1, v2 = binned_vecs[i], binned_vecs[j]
            denom = np.linalg.norm(v1) * np.linalg.norm(v2)
            same_sims.append(float(np.dot(v1, v2) / denom) if denom > 0 else 0.0)
        else:
            i, j = rng.choice(n, 2, replace=False)
            if mol_ids[i] == mol_ids[j]:
                continue
            v1, v2 = binned_vecs[i], binned_vecs[j]
            denom = np.linalg.norm(v1) * np.linalg.norm(v2)
            diff_sims.append(float(np.dot(v1, v2) / denom) if denom > 0 else 0.0)

    fig, ax = plt.subplots()
    ax.hist(same_sims, bins=40, alpha=0.6, density=True, label=f"same molecule (n={len(same_sims)})")
    ax.hist(diff_sims, bins=40, alpha=0.6, density=True, label=f"different molecule (n={len(diff_sims)})")
    ax.set_xlabel("binned cosine similarity")
    ax.set_title("Same-molecule vs different-molecule spectral similarity")
    ax.legend()
    plt.show()

    print(f"Mean same-molecule similarity: {np.mean(same_sims):.3f}" if same_sims else "No same-molecule pairs sampled.")
    print(f"Mean different-molecule similarity: {np.mean(diff_sims):.3f}" if diff_sims else "No different-molecule pairs sampled.")
else:
    print("Skipping -- requires train_spectrum_features and molecule_id_col.")


**Observation:** compare the two similarity distributions above.

**Why it matters:** if same-molecule similarity is clearly higher, nearest-neighbor
spectral retrieval is a credible baseline; if the distributions overlap heavily, retrieval
alone will be weak and structural priors (mass filtering, fragmentation rules) matter more.

**Implication for modeling:** this directly justifies (or tempers expectations for)
Notebook 02's retrieval-based approach.


## 20. Candidate-space analysis using precursor mass

In [ ]:

if len(struct_df) and "exact_mass" in struct_df.columns:
    valid = struct_df.dropna(subset=["exact_mass"]).drop_duplicates("connectivity_key").reset_index(drop=True)
    masses = np.sort(valid["exact_mass"].values)

    def count_within_ppm(masses_sorted, ppm):
        n = len(masses_sorted)
        counts = np.zeros(n, dtype=int)
        left = 0
        for i in range(n):
            tol = masses_sorted[i] * ppm / 1e6
            while masses_sorted[left] < masses_sorted[i] - tol:
                left += 1
            right = i
            while right + 1 < n and masses_sorted[right + 1] <= masses_sorted[i] + tol:
                right += 1
            counts[i] = right - left  # excluding self
        return counts

    rows = []
    for ppm in [5, 10, 20]:
        counts = count_within_ppm(masses, ppm)
        rows.append({
            "tolerance": f"±{ppm} ppm",
            "median_candidates": int(np.median(counts)),
            "p90_candidates": int(np.percentile(counts, 90)),
            "max_candidates": int(counts.max()),
        })
    candidate_space_table = pd.DataFrame(rows)
    display(candidate_space_table)
else:
    candidate_space_table = pd.DataFrame()
    print("Skipping -- requires struct_df with exact_mass.")


**Observation:** median/p90/max candidate counts at ±5/10/20 ppm mass tolerance, above.

**Why it matters:** small median candidate counts mean a `mass filter -> rerank` pipeline
is computationally practical; large tails (p90/max) mean some molecules will need a
stronger reranker to disambiguate many isobaric/isomeric candidates.

**Implication for modeling:** Notebook 02's candidate generator should use a mass window
in this range, informed by the adduct-offset analysis in Section 8.


## 21. Structural novelty / validation difficulty

In [ ]:

if mol_col and mol_col in train_df.columns and len(struct_df):
    conn_map_full = struct_df.set_index(mol_col)["connectivity_key"]
    scaffold_map_full = struct_df.set_index(mol_col)["scaffold_smiles"] if "scaffold_smiles" in struct_df.columns else None

    groups_connectivity = train_df[mol_col].map(conn_map_full)
    groups_scaffold = train_df[mol_col].map(scaffold_map_full) if scaffold_map_full is not None else None

    N_SPLITS = 5

    def summarize_split(name, tr_idx, va_idx, group_series):
        tr_groups = set(group_series.iloc[tr_idx].dropna())
        va_groups = set(group_series.iloc[va_idx].dropna())
        overlap = len(tr_groups & va_groups)
        row = {
            "split": name,
            "n_train": len(tr_idx),
            "n_val": len(va_idx),
            "n_train_groups": len(tr_groups),
            "n_val_groups": len(va_groups),
            "group_overlap": overlap,
        }
        if mz_col in train_df.columns:
            row["train_mz_mean"] = train_df[mz_col].iloc[tr_idx].mean()
            row["val_mz_mean"] = train_df[mz_col].iloc[va_idx].mean()
        if adduct_col in train_df.columns:
            row["val_adduct_only_in_val"] = len(
                set(train_df[adduct_col].iloc[va_idx].dropna())
                - set(train_df[adduct_col].iloc[tr_idx].dropna())
            )
        return row

    split_summaries = []

    # Split A: naive random
    rng = np.random.default_rng(SEED)
    idx = np.arange(len(train_df))
    rng.shuffle(idx)
    cut = int(0.2 * len(idx))
    va_idx, tr_idx = idx[:cut], idx[cut:]
    split_summaries.append(summarize_split("A: random spectrum split", tr_idx, va_idx, groups_connectivity))

    # Split B: GroupKFold by connectivity
    valid_group_mask = groups_connectivity.notna()
    if valid_group_mask.sum() > N_SPLITS:
        gkf = GroupKFold(n_splits=N_SPLITS)
        X_dummy = np.zeros(valid_group_mask.sum())
        tr_i, va_i = next(gkf.split(X_dummy, groups=groups_connectivity[valid_group_mask].values))
        orig_idx = np.where(valid_group_mask.values)[0]
        split_summaries.append(summarize_split(
            "B: GroupKFold by connectivity_key", orig_idx[tr_i], orig_idx[va_i], groups_connectivity
        ))

    # Split C: GroupKFold by scaffold
    if groups_scaffold is not None:
        valid_scaf_mask = groups_scaffold.notna()
        if valid_scaf_mask.sum() > N_SPLITS:
            gkf = GroupKFold(n_splits=N_SPLITS)
            X_dummy = np.zeros(valid_scaf_mask.sum())
            tr_i, va_i = next(gkf.split(X_dummy, groups=groups_scaffold[valid_scaf_mask].values))
            orig_idx = np.where(valid_scaf_mask.values)[0]
            split_summaries.append(summarize_split(
                "C: GroupKFold by scaffold", orig_idx[tr_i], orig_idx[va_i], groups_scaffold
            ))

    split_comparison = pd.DataFrame(split_summaries)
    display(split_comparison)

    print("\nKey check -- group_overlap should be 0 for splits B and C, and is typically > 0 for split A:")
    display(split_comparison[["split", "group_overlap"]])
else:
    split_comparison = pd.DataFrame()
    print("Skipping -- requires struct_df, molecule_id_col, and train_df.")


## 22. Recommended competition CV strategy

*(Filled in dynamically from the Section 21 results — re-run after the data-dependent
cells above execute on the real competition data.)*


In [ ]:

if len(split_comparison):
    a_overlap = split_comparison.loc[split_comparison["split"].str.startswith("A"), "group_overlap"]
    b_overlap = split_comparison.loc[split_comparison["split"].str.startswith("B"), "group_overlap"]
    c_overlap = split_comparison.loc[split_comparison["split"].str.startswith("C"), "group_overlap"]

    a_val = int(a_overlap.iloc[0]) if len(a_overlap) else None
    b_val = int(b_overlap.iloc[0]) if len(b_overlap) else None
    c_val = int(c_overlap.iloc[0]) if len(c_overlap) else None

    print("RECOMMENDATION (derived from the measured overlaps above):")
    print("-" * 70)
    if a_val and a_val > 0:
        print(f"- Split A (naive random) leaks {a_val} molecule groups into validation -> DO NOT TRUST.")
    print(f"- Split B (GroupKFold by connectivity_key) group_overlap = {b_val} -> "
          f"{'safe, use as PRIMARY CV.' if b_val == 0 else 'unexpected overlap, investigate grouping.'}")
    if c_val is not None:
        print(f"- Split C (GroupKFold by scaffold) group_overlap = {c_val} -> "
              f"{'safe, use as SECONDARY/HARD CV for novel-scaffold generalization.' if c_val == 0 else 'unexpected overlap, investigate.'}")
    print("""
Primary CV     : GroupKFold(n_splits=5) grouped by connectivity_key (InChIKey14 after
                 tautomer canonicalization). Every molecule/connectivity appears in
                 exactly one fold.
Secondary CV   : GroupKFold grouped by Murcko scaffold, used to stress-test
                 generalization to unseen chemotypes (harder, smaller effective
                 training set per fold).
DO NOT TRUST   : any random spectrum-level split -- Section 6C and Split A above show
                 it leaks molecule identity between train and validation and will
                 overstate MRR@25 relative to true held-out performance.
""")
else:
    print("Run Section 21 on real data first to auto-derive this recommendation.")


## 23. Estimate known-vs-novel difficulty

Using the connectivity-grouped fold from Section 21, measure each validation molecule's
maximum Tanimoto similarity to anything in that fold's training partition.


In [ ]:

if len(valid_struct) and len(split_comparison) and mol_col in train_df.columns:
    # Reuse the connectivity-grouped split from Section 21
    valid_group_mask = groups_connectivity.notna()
    gkf = GroupKFold(n_splits=5)
    X_dummy = np.zeros(valid_group_mask.sum())
    tr_i, va_i = next(gkf.split(X_dummy, groups=groups_connectivity[valid_group_mask].values))
    orig_idx = np.where(valid_group_mask.values)[0]
    tr_idx, va_idx = orig_idx[tr_i], orig_idx[va_i]

    tr_conn = set(groups_connectivity.iloc[tr_idx].dropna())
    va_conn = set(groups_connectivity.iloc[va_idx].dropna())

    fp_lookup = dict(zip(valid_struct["connectivity_key"], valid_struct["morgan_fp"]))
    train_fps = [fp_lookup[c] for c in tr_conn if c in fp_lookup and fp_lookup[c] is not None]

    max_sims = []
    va_conn_list = [c for c in va_conn if c in fp_lookup and fp_lookup[c] is not None]
    for c in va_conn_list:
        fp = fp_lookup[c]
        if fp is None or not train_fps:
            continue
        sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
        max_sims.append(max(sims) if sims else 0.0)

    max_sims = np.array(max_sims)
    if len(max_sims):
        fig, ax = plt.subplots()
        ax.hist(max_sims, bins=40)
        ax.set_xlabel("max Tanimoto similarity to any training structure")
        ax.set_ylabel("count of validation structures")
        ax.set_title("Validation structural novelty (connectivity-grouped fold)")
        plt.show()

        bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
        labels = ["<0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
        bucketed = pd.cut(max_sims, bins=bins, labels=labels, include_lowest=True)
        bucket_counts = bucketed.value_counts().sort_index()
        bucket_pct = (bucket_counts / bucket_counts.sum() * 100).round(1)
        display(pd.DataFrame({"count": bucket_counts, "pct": bucket_pct}))
    else:
        print("Not enough fingerprints to compute max-Tanimoto-to-train.")
else:
    print("Skipping -- requires valid_struct (Section 15) and split_comparison (Section 21).")


**Observation:** distribution of validation molecules' maximum similarity to any
training molecule, bucketed exploratorily (bins are not scientifically load-bearing).

**Why it matters:** a large low-similarity mass means the grouped fold genuinely tests
generalization to novel chemistry, matching the real test set's likely composition.

**Implication for modeling:** report future model performance broken out by these buckets,
not just a single aggregate MRR@25.


## 24. Train/test distribution shift

In [ ]:

shift_rows = []

def ks_shift(train_vals, test_vals):
    train_vals = pd.Series(train_vals).dropna()
    test_vals = pd.Series(test_vals).dropna()
    if len(train_vals) < 5 or len(test_vals) < 5:
        return np.nan
    stat, _ = sp_stats.ks_2samp(train_vals, test_vals)
    return stat

def concern_level(stat):
    if pd.isna(stat):
        return "n/a"
    if stat < 0.05:
        return "small"
    if stat < 0.15:
        return "moderate"
    return "large"

for col, label in [(mz_col, "precursor_mz"), (ce_col, "collision_energy")]:
    if col and col in train_df.columns and col in test_df.columns and train_df[col].dtype.kind in "if":
        stat = ks_shift(train_df[col], test_df[col])
        shift_rows.append({"variable": label, "ks_statistic": round(stat, 4) if pd.notna(stat) else np.nan,
                            "concern_level": concern_level(stat),
                            "comment": "KS distance between train/test empirical distributions"})

for col, label in [(adduct_col, "adduct"), (ion_col, "ion_mode")]:
    if col and col in train_df.columns and col in test_df.columns:
        tr_p = train_df[col].value_counts(normalize=True)
        te_p = test_df[col].value_counts(normalize=True)
        all_cats = tr_p.index.union(te_p.index)
        tr_p = tr_p.reindex(all_cats, fill_value=0)
        te_p = te_p.reindex(all_cats, fill_value=0)
        tvd = 0.5 * (tr_p - te_p).abs().sum()  # total variation distance
        shift_rows.append({"variable": label, "ks_statistic": round(tvd, 4),
                            "concern_level": concern_level(tvd),
                            "comment": "total variation distance between category proportions"})

if 'peak_feat_df' in dir() and len(peak_feat_df) and 'test_peak_feat_df' in dir():
    for col in ["n_peaks", "fragment_mass_range"]:
        stat = ks_shift(peak_feat_df[col], test_peak_feat_df[col])
        shift_rows.append({"variable": col, "ks_statistic": round(stat, 4) if pd.notna(stat) else np.nan,
                            "concern_level": concern_level(stat), "comment": "KS distance, spectral feature"})

shift_table = pd.DataFrame(shift_rows)
shift_table


## 25. Missing values and data quality audit

We only **flag** suspicious records here — nothing is silently dropped.


In [ ]:

quality_flags = {}

if mz_col in train_df.columns:
    quality_flags["missing_precursor_mz"] = train_df[mz_col].isna().sum()
if DATA_MAP["peaks_col"] in train_df.columns:
    empty_peaks = train_df.apply(lambda r: len(get_spectrum_peaks(r)[0]) == 0, axis=1)
    quality_flags["empty_peak_lists"] = int(empty_peaks.sum())

    n_low_peak = train_df.apply(lambda r: 0 < len(get_spectrum_peaks(r)[0]) <= 2, axis=1).sum()
    quality_flags["spectra_with_1_or_2_peaks"] = int(n_low_peak)

    def has_zero_or_negative_intensity(row):
        _, inten = get_spectrum_peaks(row)
        return len(inten) > 0 and (inten <= 0).any()
    quality_flags["spectra_with_zero_or_negative_intensity"] = int(train_df.apply(has_zero_or_negative_intensity, axis=1).sum())

    def has_unsorted_mz(row):
        mz, _ = get_spectrum_peaks(row)
        return len(mz) > 1 and not np.all(np.diff(mz) >= 0)
    quality_flags["spectra_with_unsorted_mz"] = int(train_df.apply(has_unsorted_mz, axis=1).sum())

    def has_duplicate_peaks(row):
        mz, _ = get_spectrum_peaks(row)
        return len(mz) != len(set(np.round(mz, 4)))
    quality_flags["spectra_with_duplicate_peaks"] = int(train_df.apply(has_duplicate_peaks, axis=1).sum())

    if mz_col in train_df.columns:
        def frag_above_precursor(row):
            mz, _ = get_spectrum_peaks(row)
            prec = row.get(mz_col)
            return len(mz) > 0 and pd.notna(prec) and (mz > prec + 0.5).any()
        quality_flags["spectra_with_fragment_above_precursor"] = int(train_df.apply(frag_above_precursor, axis=1).sum())

if len(struct_df):
    quality_flags["invalid_smiles"] = int(struct_df["canonical_smiles"].isna().sum())

if mol_col in train_df.columns:
    quality_flags["duplicated_ids"] = int(train_df[mol_col].duplicated().sum())
    if spec_col and spec_col in train_df.columns:
        quality_flags["duplicated_spectrum_ids"] = int(train_df[spec_col].duplicated().sum())

if adduct_col in train_df.columns:
    known_adducts = set(ADDUCT_OFFSETS.keys())
    quality_flags["adduct_values_not_in_known_list"] = int((~train_df[adduct_col].isin(known_adducts)).sum())

quality_report = pd.Series(quality_flags, name="flagged_count").to_frame()
quality_report["flagged_pct_of_rows"] = (quality_report["flagged_count"] / max(len(train_df), 1) * 100).round(2)
quality_report


## 26. Submission inspection

In [ ]:

sample_sub = sample_submission
print("sample_submission shape:", sample_sub.shape)
if len(sample_sub):
    display(sample_sub.head())
    print("columns:", list(sample_sub.columns))


In [ ]:

def validate_submission(df, id_col=None, pred_col=None, expected_ids=None, max_candidates=25):
    """Validate a candidate submission dataframe against the CASMI 2026 format.
    Returns (is_valid: bool, issues: list[str]).
    """
    issues = []
    if id_col is None or id_col not in df.columns:
        issues.append(f"Missing expected id column ({id_col!r} not found). Columns present: {list(df.columns)}")
        return False, issues
    if pred_col is None or pred_col not in df.columns:
        issues.append(f"Missing expected prediction column ({pred_col!r} not found). Columns present: {list(df.columns)}")
        return False, issues

    if df[id_col].isna().any():
        issues.append(f"{df[id_col].isna().sum()} null id(s) in {id_col}.")
    if df[pred_col].isna().any():
        issues.append(f"{df[pred_col].isna().sum()} null prediction(s) in {pred_col}.")
    dup_ids = df[id_col].duplicated().sum()
    if dup_ids:
        issues.append(f"{dup_ids} duplicated id(s) in {id_col}.")

    if expected_ids is not None:
        missing_ids = set(expected_ids) - set(df[id_col])
        extra_ids = set(df[id_col]) - set(expected_ids)
        if missing_ids:
            issues.append(f"{len(missing_ids)} expected id(s) missing from submission.")
        if extra_ids:
            issues.append(f"{len(extra_ids)} unexpected id(s) present in submission.")

    def _check_row(preds):
        if pd.isna(preds) or str(preds).strip() == "":
            return "empty prediction string"
        candidates = [c for c in str(preds).split(";") if c.strip() != ""]
        if len(candidates) == 0:
            return "no candidates after splitting on ';'"
        if len(candidates) > max_candidates:
            return f"{len(candidates)} candidates exceeds max of {max_candidates}"
        return None

    row_issues = df[pred_col].apply(_check_row)
    n_bad_rows = row_issues.notna().sum()
    if n_bad_rows:
        issues.append(f"{n_bad_rows} row(s) with malformed candidate strings (see examples).")

    is_valid = len(issues) == 0
    return is_valid, issues

print("validate_submission() ready.")

if len(sample_sub):
    id_guess = DATA_MAP["molecule_id_col"] if DATA_MAP["molecule_id_col"] in sample_sub.columns else sample_sub.columns[0]
    pred_guess = "smiles" if "smiles" in sample_sub.columns else (sample_sub.columns[1] if len(sample_sub.columns) > 1 else None)
    ok, issues = validate_submission(sample_sub, id_col=id_guess, pred_col=pred_guess,
                                      expected_ids=test_df[mol_col] if mol_col in test_df.columns else None)
    print(f"sample_submission valid: {ok}")
    for issue in issues:
        print(" -", issue)


## 27. Important visualizations — recap

The high-value plots requested for this notebook were produced in-line, close to their
analysis, rather than repeated here:

1. Train/test precursor-mass distribution — Section 8 / Section 4
2. Spectra-per-molecule distribution — Section 6B
3. Number-of-peaks distribution — Section 12
4. Ion mode and adduct frequencies — Section 9
5. Representative spectrum plots — Section 13
6. Same molecule under multiple acquisition settings — Section 14
7. Near-identical mass but different structures — Section 16
8. Same-vs-different molecule spectral similarity — Section 19
9. Scaffold frequency distribution — Section 7
10. Validation max-Tanimoto-to-train distribution — Section 23
11. Candidate counts after mass filtering — Section 20 (table)


## 28. Final EDA summary — What We Learned

In [ ]:

def _fmt(x, nd=3):
    try:
        return f"{x:,.{nd}f}"
    except Exception:
        return str(x)

summary_lines = []
summary_lines.append("### Dataset structure")
summary_lines.append(
    f"- OBSERVATION UNIT = spectrum, GROUPING UNIT = molecule/connectivity, TARGET = SMILES.\n"
    f"- Train rows: {len(train_df)}, unique molecules: {safe_nunique(train_df, mol_col)}, "
    f"unique connectivity keys: {struct_df['connectivity_key'].nunique() if len(struct_df) else 'n/a'}."
)

summary_lines.append("\n### Spectral characteristics")
if 'peak_feat_df' in dir() and len(peak_feat_df):
    summary_lines.append(
        f"- Median peaks/spectrum: {_fmt(peak_feat_df['n_peaks'].median(), 1)}, "
        f"median fragment mass range: {_fmt(peak_feat_df['fragment_mass_range'].median())}."
    )

summary_lines.append("\n### Chemical structure diversity")
if len(struct_df):
    summary_lines.append(
        f"- {struct_df['connectivity_key'].nunique()} unique structures across "
        f"{struct_df['scaffold_smiles'].nunique() if 'scaffold_smiles' in struct_df.columns else 'n/a'} scaffolds."
    )

summary_lines.append("\n### Leakage risk")
if len(split_comparison):
    a_row = split_comparison[split_comparison['split'].str.startswith('A')]
    if len(a_row):
        summary_lines.append(
            f"- Naive random spectrum split showed group_overlap = {int(a_row['group_overlap'].iloc[0])} "
            "-> random CV is NOT trustworthy for this dataset."
        )

summary_lines.append("\n### Train/test shift")
if len(shift_table):
    concerning = shift_table[shift_table['concern_level'] == 'large']
    if len(concerning):
        summary_lines.append(f"- Large shift flagged for: {', '.join(concerning['variable'])}.")
    else:
        summary_lines.append("- No variable showed a 'large' shift by the KS/TVD heuristics used.")

summary_lines.append("\n### Candidate-generation opportunity")
if len(candidate_space_table):
    summary_lines.append(
        f"- At ±10 ppm, median candidate count = "
        f"{candidate_space_table.set_index('tolerance').loc['±10 ppm', 'median_candidates'] if '±10 ppm' in candidate_space_table['tolerance'].values else 'n/a'}."
    )

summary_lines.append("\n### Retrieval opportunity")
if 'same_sims' in dir() and same_sims and diff_sims:
    summary_lines.append(
        f"- Mean same-molecule similarity ({np.mean(same_sims):.3f}) vs "
        f"different-molecule similarity ({np.mean(diff_sims):.3f})."
    )

summary_lines.append("\n### Novel-structure challenge")
if 'max_sims' in dir() and len(max_sims):
    frac_novel = np.mean(max_sims < 0.4)
    summary_lines.append(f"- {frac_novel:.1%} of connectivity-grouped validation structures have "
                          "max Tanimoto < 0.4 to any training structure.")

summary_lines.append("\n### Validation recommendation")
summary_lines.append(
    "- PRIMARY: GroupKFold by connectivity_key (InChIKey14, tautomer-canonicalized).\n"
    "- SECONDARY / HARD: GroupKFold by Murcko scaffold.\n"
    "- DO NOT TRUST: naive random spectrum-level splits."
)

print("\n".join(summary_lines))


## 29. Output reusable artifacts

Persist clean, deduplicated tables for Notebook 02.


In [ ]:

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if len(struct_df):
    n_spectra_per_mol = (
        train_df.groupby(mol_col).size().rename("n_spectra") if mol_col in train_df.columns else None
    )
    structure_metadata = struct_df.drop_duplicates("connectivity_key").copy()
    if n_spectra_per_mol is not None and mol_col in structure_metadata.columns:
        structure_metadata = structure_metadata.merge(n_spectra_per_mol, on=mol_col, how="left")

    keep_cols = [c for c in [
        mol_col, "canonical_smiles", "connectivity_key", "scaffold_smiles", "exact_mass",
        "molecular_formula", "num_atoms", "num_rings", "n_spectra"
    ] if c and c in structure_metadata.columns]
    structure_metadata_out = structure_metadata[keep_cols]
    structure_metadata_out.to_parquet(OUT_DIR / "structure_metadata.parquet", index=False)
    print(f"Saved structure_metadata.parquet: {structure_metadata_out.shape}")

if len(train_spectrum_features):
    spec_keep_cols = [c for c in [
        spec_col, mol_col, mz_col, adduct_col, ion_col, ce_col,
        "n_peaks", "base_peak_mz"
    ] if c and c in train_spectrum_features.columns]
    spectrum_metadata_out = train_spectrum_features[spec_keep_cols]
    spectrum_metadata_out.to_parquet(OUT_DIR / "spectrum_metadata.parquet", index=False)
    print(f"Saved spectrum_metadata.parquet: {spectrum_metadata_out.shape}")

if len(struct_df):
    molecule_metadata_out = structure_metadata_out if 'structure_metadata_out' in dir() else struct_df
    molecule_metadata_out.to_parquet(OUT_DIR / "molecule_metadata.parquet", index=False)
    print(f"Saved molecule_metadata.parquet: {molecule_metadata_out.shape}")

print("\nArtifacts written to /kaggle/working/ (only when the dependent cells above "
      "have real data to work with).")


## 30–31. Coding requirements & constraints — recap

This notebook intentionally does **not**:
- train a deep-learning model or Transformer,
- generate SMILES candidates,
- build a full candidate database,
- use test labels or infer hidden test information,
- treat a naive random validation split as trustworthy,
- discard duplicate spectra before studying them,
- assume one spectrum <-> one unique molecule,
- assume precursor mass uniquely determines a molecule,
- overinterpret chemistry patterns without direct evidence in the data.

Reusable functions defined above (usable as-is from Notebook 02):
`load_spectrum_file`, `parse_peak_string`, `load_full_tabular`, `canonicalize_smiles`,
`get_connectivity_key`, `compute_molecule_descriptors`, `get_murcko_scaffold`,
`get_morgan_fp`, `plot_spectrum`, `compute_spectrum_features`, `bin_spectrum`,
`compute_spectrum_similarity`, `validate_submission`.


## 32. Final deliverable — handoff to Notebook 02

**Notebook 02 — Leakage-Safe Spectral Retrieval Baseline** should implement:

```
spectrum preprocessing
  -> mass-compatible candidate filtering   (Section 20: mass window from Section 8's
                                             adduct-offset analysis)
  -> binned spectral representations       (Section 19's bin_spectrum())
  -> cosine / peak-based spectral similarity (Section 19's compute_spectrum_similarity())
  -> nearest-neighbor retrieval over `structure_metadata.parquet`
  -> Top-25 ranking
  -> MRR@25, evaluated under the PRIMARY (connectivity-grouped) and SECONDARY
     (scaffold-grouped) CV from Section 21/22, broken out by the novelty buckets
     from Section 23
```

Everything Notebook 02 needs — grouping keys, mass tolerances, similarity function,
validation folds, and quality flags — was derived directly from the data in this
notebook, not assumed in advance.
